In [1]:
import os
import json
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [2]:
def load_sweep_results_with_hyperparams(sweep_ids: list, wandb_dir: Path) -> pd.DataFrame:
    """Load all run results including hyperparameters from multiple sweeps."""
    all_results = []
    
    for sweep_id in sweep_ids:
        sweep_dir = wandb_dir / f"sweep-{sweep_id}"
        if not sweep_dir.exists():
            print(f"Warning: Sweep directory not found for {sweep_id}")
            continue
            
        print(f"Loading sweep: {sweep_id}")
        
        # Get all run IDs from sweep config files
        for config_file in sweep_dir.glob("config-*.yaml"):
            run_id = config_file.stem.replace("config-", "")
            
            # Find the corresponding run directory
            run_dirs = list(wandb_dir.glob(f"run-*-{run_id}"))
            if not run_dirs:
                continue
            
            run_dir = run_dirs[0]
            
            # Load config from sweep directory
            with open(config_file) as f:
                config = yaml.safe_load(f)
            
            # Load summary from run directory
            summary_file = run_dir / "files" / "wandb-summary.json"
            if not summary_file.exists():
                continue
                
            with open(summary_file) as f:
                summary = json.load(f)
            
            # Helper to extract value from config
            def get_config_value(key):
                val = config.get(key, {})
                if isinstance(val, dict):
                    return val.get("value")
                return val
            
            # Extract all relevant info including hyperparameters
            result = {
                "sweep_id": sweep_id,
                "run_id": run_id,
                "setting": get_config_value("setting"),
                "method": get_config_value("method"),
                "seed": get_config_value("seed"),
                "lr": get_config_value("lr"),
                "beta_efc": get_config_value("beta_efc"),
                "target_lr": get_config_value("target_lr"),
                "batch_size": get_config_value("batch_size"),
                "epochs": get_config_value("epochs"),
                "final_avg_accuracy": summary.get("final_avg_accuracy"),
                "final_forgetting": summary.get("final_forgetting"),
                "layer_size": get_config_value("layer_size"),
                "cnn_pretrained": get_config_value("cnn_pretrained")
            }
            all_results.append(result)
    
    return pd.DataFrame(all_results)

# Load sweeps

In [3]:
import wandb

WANDB_DIR = Path("./wandb")

api = wandb.Api()
runs = api.runs("equilibrium-fisher-control/icml-final-runs")

sweep_dict = {}
for run in runs:
    if run.sweep:
        sweep_dict[run.sweep.name] = run.sweep.id

SWEEP_DICTS = sweep_dict
print(SWEEP_DICTS)
print(len(SWEEP_DICTS))


{'bp_taskIL_MNIST': 'hpaqt9uu', 'oewc_taskIL_MNIST': '7rq9cqz0', 'taskIL_MNIST_EFC': 'vxugh8x8', 'derpp_taskIL_MNIST': '32ir9287', 'bp_taskIL_MNIST_Encoded': 'lzrx6hu9', 'oewc_taskIL_MNIST_Encoded': 'zq46it3o', 'taskIL_EFC_MNIST_Encoded': '0hp00l2w', 'bp_taskIL_CIFAR10': 'gsweluk8', 'oewc_taskIL_CIFAR10': 'p96byc60', 'bp_taskIL_TinyImageNet': '2o7y9ls0', 'oewc_taskIL_TinyImageNet': '7zwzp579', 'derpp_taskIL_MNIST_Encoded': '7jkr4mwn', 'bp_classIL_MNIST': 'biqtavdj', 'oewc_classIL_MNIST': 'zz4qngi2', 'bp_classIL_MNIST_Encoded': 's3e48hem', 'oewc_classIL_MNIST_Encoded': '105erujz', 'bp_classIL_CIFAR5Task': 'yfn4e6a7', 'derpp_taskIL_CIFAR10': 'kx3jfs1s', 'oewc_classIL_CIFAR5Task': 'ze3yunr1', 'bp_classIL_TinyImageNet': 'tleto4p7', 'oewc_classIL_TinyImageNet': 'rdyx9rzx', 'derpp_taskIL_TinyImageNet': 'b29xml5r', 'ewc_taskIL_MNIST': 'gydss951', 'si_taskIL_MNIST': '0z9jwn2l', 'ewc_taskIL_MNIST_Encoded': 'ri00yzbr', 'taskIL_EFC_CIFAR10': 'ntxnv33s', 'si_taskIL_MNIST_Encoded': 'a81okinj', 'ewc

# Single Sweep Analysis

In [4]:
# Summary statistics

wanted_sweep = "2y74n3t7"

df = load_sweep_results_with_hyperparams(sweep_ids=[wanted_sweep], wandb_dir=WANDB_DIR)

avg_acc = df['final_avg_accuracy'].mean()
std_acc = df['final_avg_accuracy'].std()
n_seeds = df['seed'].nunique()

print(f"=== Summary for {wanted_sweep} ===")
print(f"Average Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}%")
print(f"Number of seeds: {n_seeds}")
print()

# Performance of all models in the sweep
print(f"=== Individual Run Performance ===")
performance_df = df[['seed', 'method', 'setting', 'lr', 'beta_efc', 'target_lr', 'final_avg_accuracy']].copy()
performance_df = performance_df.sort_values('seed')
performance_df['final_avg_accuracy'] = performance_df['final_avg_accuracy'].round(2)
print(performance_df.to_string(index=False))

Loading sweep: 2y74n3t7
=== Summary for 2y74n3t7 ===
Average Accuracy: 28.96% ± 0.84%
Number of seeds: 10

=== Individual Run Performance ===
 seed method            setting     lr  beta_efc  target_lr  final_avg_accuracy
    0    efc TaskILTinyImageNet 0.0001       0.1        0.1               27.55
    1    efc TaskILTinyImageNet 0.0001       0.1        0.1               29.71
    2    efc TaskILTinyImageNet 0.0001       0.1        0.1               28.29
    3    efc TaskILTinyImageNet 0.0001       0.1        0.1               29.91
    4    efc TaskILTinyImageNet 0.0001       0.1        0.1               27.83
    5    efc TaskILTinyImageNet 0.0001       0.1        0.1               28.66
    6    efc TaskILTinyImageNet 0.0001       0.1        0.1               29.50
    7    efc TaskILTinyImageNet 0.0001       0.1        0.1               29.32
    8    efc TaskILTinyImageNet 0.0001       0.1        0.1               29.07
    9    efc TaskILTinyImageNet 0.0001       0.1        0.

## Other Methods 

In [11]:
# Dictionary mapping sweep names to their IDs for "other" methods

for sweep in SWEEP_DICTS.keys():
    try:
        wanted_sweep = sweep
        df = load_sweep_results_with_hyperparams([SWEEP_DICTS[wanted_sweep]], WANDB_DIR)

        # Summary statistics grouped by method
        print(f"=== Summary for {wanted_sweep} ===\n")
        summary = df.groupby('method').agg({
            'final_avg_accuracy': ['mean', 'std', 'count'],
        }).round(2)
        summary.columns = ['Acc Mean', 'Acc Std', 'N Seeds']
        print(summary.to_string())

        print(f"\n=== Individual Run Performance ===")
        # performance_df = df[['seed', 'method', 'setting', 'final_avg_accuracy']].copy()
        # performance_df = performance_df.sort_values(['method', 'seed'])
        # performance_df['final_avg_accuracy'] = performance_df['final_avg_accuracy'].round(2)
        # print(performance_df.to_string(index=False))
    except:
        continue


Loading sweep: hpaqt9uu
=== Summary for bp_taskIL_MNIST ===

        Acc Mean  Acc Std  N Seeds
method                            
bp         98.21     0.09        5

=== Individual Run Performance ===
Loading sweep: 7rq9cqz0
=== Summary for oewc_taskIL_MNIST ===

        Acc Mean  Acc Std  N Seeds
method                            
oewc       99.03     0.19        5

=== Individual Run Performance ===
Loading sweep: vxugh8x8
=== Summary for taskIL_MNIST_EFC ===

        Acc Mean  Acc Std  N Seeds
method                            
efc        95.97      4.3        5

=== Individual Run Performance ===
Loading sweep: 32ir9287
=== Summary for derpp_taskIL_MNIST ===

        Acc Mean  Acc Std  N Seeds
method                            
derpp      99.02     0.04        5

=== Individual Run Performance ===
Loading sweep: lzrx6hu9
=== Summary for bp_taskIL_MNIST_Encoded ===

        Acc Mean  Acc Std  N Seeds
method                            
bp         89.99     5.47        5

=== Individ

In [7]:
# Build results table from all sweeps
# Sweep names follow pattern: {method}_{setting}_{dataset} or {setting}_{method}_{dataset}

# Define the table structure
datasets = ["MNIST", "MNIST_Encoded", "CIFAR10", "TinyImageNet"]
settings = ["taskIL", "classIL"]
methods = ["BP", "EWC", "oEWC", "SI", "EFC", "DERPP"]

# Method name mapping (lowercase -> display name)
method_map = {"bp": "BP", "ewc": "EWC", "oewc": "oEWC", "si": "SI", "efc": "EFC", "derpp": "DERPP"}

# Dataset name mapping (CIFAR5Task and CIFAR10 are the same)
dataset_map = {"CIFAR5Task": "CIFAR10"}

# Setting name mapping (normalize case)
setting_map = {"classil": "classIL", "taskil": "taskIL"}

# Create multi-level columns: (setting, dataset)
columns = pd.MultiIndex.from_product([settings, datasets], names=["Setting", "Dataset"])
results_df = pd.DataFrame(index=methods, columns=columns)

# Parse sweep names and load results
for sweep_name, sweep_id in SWEEP_DICTS.items():
    # Parse the sweep name to extract method, setting, dataset
    parts = sweep_name.split("_")
    
    # Handle both naming conventions:
    # "bp_taskIL_MNIST" -> method=bp, setting=taskIL, dataset=MNIST
    # "taskIL_MNIST_EFC" -> setting=taskIL, dataset=MNIST, method=EFC
    if parts[0].lower() in method_map:
        # Format: method_setting_dataset
        method_key = parts[0].lower()
        setting = parts[1]
        dataset = "_".join(parts[2:])
    else:
        # Format: setting_method_dataset or setting_dataset_method
        setting = parts[0]
        if parts[-1].upper() in ["EFC"]:
            method_key = parts[-1].lower()
            dataset = "_".join(parts[1:-1])
        else:
            method_key = parts[1].lower()
            dataset = "_".join(parts[2:])
    
    method_display = method_map.get(method_key, method_key.upper())
    
    # Normalize dataset name (CIFAR5Task -> CIFAR10)
    dataset = dataset_map.get(dataset, dataset)
    
    # Normalize setting name (ClassIL -> classIL)
    setting = setting_map.get(setting.lower(), setting)
    
    # Skip if dataset not in our list
    if dataset not in datasets:
        print(f"Skipping {sweep_name}: dataset '{dataset}' not recognized")
        continue
    
    # Load the sweep data
    df = load_sweep_results_with_hyperparams([sweep_id], WANDB_DIR)
    
    if df.empty:
        print(f"No data for {sweep_name}")
        continue
    
    # Calculate mean ± std
    mean_acc = df['final_avg_accuracy'].mean()
    std_acc = df['final_avg_accuracy'].std()
    n_runs = len(df)
    
    # Store in results table
    results_df.loc[method_display, (setting, dataset)] = f"{mean_acc:.2f} ± {std_acc:.2f}"
    print(f"{sweep_name}: {method_display} | {setting} | {dataset} -> {mean_acc:.2f} ± {std_acc:.2f} (n={n_runs})")

print("\n" + "="*80)
print("RESULTS TABLE")
print("="*80)
print(results_df.to_string())
# Convert to CSV 

Loading sweep: hpaqt9uu
bp_taskIL_MNIST: BP | taskIL | MNIST -> 98.21 ± 0.09 (n=5)
Loading sweep: 7rq9cqz0
oewc_taskIL_MNIST: oEWC | taskIL | MNIST -> 99.03 ± 0.19 (n=5)
Loading sweep: vxugh8x8
taskIL_MNIST_EFC: EFC | taskIL | MNIST -> 95.97 ± 4.30 (n=5)
Loading sweep: 32ir9287
derpp_taskIL_MNIST: DERPP | taskIL | MNIST -> 99.02 ± 0.04 (n=5)
Loading sweep: lzrx6hu9
bp_taskIL_MNIST_Encoded: BP | taskIL | MNIST_Encoded -> 89.99 ± 5.47 (n=5)
Loading sweep: zq46it3o
oewc_taskIL_MNIST_Encoded: oEWC | taskIL | MNIST_Encoded -> 97.73 ± 2.58 (n=5)
Loading sweep: 0hp00l2w
taskIL_EFC_MNIST_Encoded: EFC | taskIL | MNIST_Encoded -> 87.35 ± 11.49 (n=5)
Loading sweep: gsweluk8
bp_taskIL_CIFAR10: BP | taskIL | CIFAR10 -> 95.76 ± 0.24 (n=5)
Loading sweep: p96byc60
oewc_taskIL_CIFAR10: oEWC | taskIL | CIFAR10 -> 95.94 ± 0.41 (n=5)
Loading sweep: 2o7y9ls0
bp_taskIL_TinyImageNet: BP | taskIL | TinyImageNet -> 28.01 ± 0.36 (n=5)
Loading sweep: 7zwzp579
oewc_taskIL_TinyImageNet: oEWC | taskIL | TinyImageNe

In [8]:
# Save results to CSV
results_df.to_csv("paper_results.csv")
print("Saved to paper_results.csv")

Saved to paper_results.csv
